# 16_slide_figures.ipynb — สร้างกราฟและตารางในสไลด์สัมมนาจากไฟล์ผลจริง

notebook นี้**ไม่เทรนโมเดลใหม่** อ่านผลที่ notebook 11–15 บันทึกไว้ใน `outputs/` แล้ววาดกราฟแต่ละรูปในสไลด์ใหม่
ทุกตัวเลขในสไลด์จึงตรวจย้อนกลับได้ว่ามาจากไฟล์ไหน ผลทั้งหมดบันทึกที่ `outputs/16_slide_figures/`

| สไลด์ | เนื้อหา | ไฟล์ข้อมูลต้นทาง | notebook ที่สร้างข้อมูล | หัวข้อในนี้ |
|---|---|---|---|---|
| Dataset | 99,340 encounter, 11.4% / 35.7% / 52.9% | `data/diabetic_data.csv` | — (คำนวณใหม่ที่นี่) | 1 |
| วิธีการ 1 | train 63,605 / val 15,962 / test 19,773 | `outputs/split_assignment.csv` | 11 | 2 |
| ผลที่ 1 | ตาราง Stacking 30 วัน vs ทุกช่วงเวลา | `outputs/11_30d/metrics.json`, `outputs/11_any/metrics.json` | 11 | 3 |
| ผลที่ 2 | Odds ratio 6 ปัจจัย | `outputs/12_features_any/B_odds_ratios.csv` | 12 | 4 |
| ผลที่ 3 | Hazard ratio ช่วง 30 วันแรก vs ระยะยาว | `outputs/13_survival/4_ph_check.csv` (+ CI ที่เพิ่มในนี้) | 13 | 5 |
| ผลที่ 4 | Ablation A–D ทั้งสองนิยาม | `outputs/14_lstm_*/test_metrics.csv` | 14 | 6 |
| แบบจำลองสุดท้าย | ROC-AUC 6 โมเดล + 95% CI | `outputs/15_final_*/test_metrics.csv`, `outputs/13_survival/summary.json`, `outputs/14_lstm_*/test_predictions.csv` | 13, 14, 15 | 7 |
| ระดับความเสี่ยง | อัตรากลับมาจริงรายระดับ | `outputs/15_final_*/risk_tiers.csv` | 15 | 8 |
| ความเป็นธรรม | อายุ 90+ เทียบผู้ป่วยทั้งหมด | `outputs/15_final_*/fairness_age.csv`, `test_metrics.csv` | 15 | 9 |
| Dashboard | ตรวจผู้ป่วย 255/259 ราย | `dashboard/validate_dashboard.py` | — | (ไฟล์แยก) |

หัวข้อ 10 ตรวจตัวเลขที่เขียนในสไลด์เทียบกับไฟล์ทั้งหมดอัตโนมัติ

**สิ่งที่ notebook นี้เพิ่มใหม่** (เดิมไม่มีในไฟล์ใด):
- 95% CI ของ hazard ratio ในสไลด์ผลที่ 3 (cluster bootstrap ตามผู้ป่วย) — เดิม `4_ph_check.csv` มีแต่ค่า HR
- 95% CI ของ ROC-AUC โมเดลสุดท้าย — เดิมอยู่แค่ในข้อความ output ของ notebook 15 ไม่ได้บันทึกเป็นไฟล์
- สัดส่วน label ของ dataset และจำนวนในแต่ละ split

## 0. Config

In [ ]:
import os, re, json, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)   # expm1 overflow ใน GroupedPH (เหมือน notebook 13)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.metrics import roc_auc_score

OUT = "../outputs"
FIG_DIR = f"{OUT}/16_slide_figures"
RAW_PATH = "../data/diabetic_data.csv"
RANDOM_STATE = 42
N_BOOT_HR = 200       # bootstrap ของ hazard ratio (หัวข้อ 5) — ใช้เวลานานที่สุดใน notebook นี้ ลดเป็น 50 ถ้าอยากเร็ว
N_BOOT_TEST = 500     # เท่ากับ notebook 15 → ได้ CI เดียวกับที่ notebook 15 พิมพ์ไว้
os.makedirs(FIG_DIR, exist_ok=True)

# ฟอนต์ไทย: ใช้ตัวแรกที่มีในเครื่อง (Windows มี Tahoma / Leelawadee UI)
THAI_FONTS = ["Tahoma", "Leelawadee UI", "TH Sarabun New", "Noto Sans Thai", "Loma", "Garuda"]
have = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams["font.family"] = [f for f in THAI_FONTS if f in have] + ["DejaVu Sans"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
print("ฟอนต์ที่ใช้:", plt.rcParams["font.family"][0])

C_ANY, C_30D = "#e48bb4", "#a8336a"     # สีเดียวกับสไลด์: ทุกช่วงเวลา (อ่อน), 30 วัน (เข้ม)
LABEL_TH = {"any": "ทุกช่วงเวลา", "30d": "30 วัน"}

def save(fig, name):
    fig.savefig(f"{FIG_DIR}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

def r2(v):   # ปัดครึ่งขึ้นแบบที่ใช้ในสไลด์ (1.045 → 1.05)
    return f"{np.floor(v * 100 + 0.5) / 100:.2f}"

def read(path, **kw):
    return pd.read_csv(f"{OUT}/{path}", **kw)

## 1. สไลด์ Dataset — สัดส่วนการกลับมารักษาซ้ำ
ทำความสะอาดแบบเดียวกับ `load_prepared()` (ตัด gender ไม่ระบุ, ตัด discharge 11/13/14/19/20/21)

In [ ]:
raw = pd.read_csv(RAW_PATH, keep_default_na=False, na_values=["?"], low_memory=False)
n_raw = len(raw)
df = raw[raw["gender"] != "Unknown/Invalid"]
df = df[~df["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]
dist = (df["readmitted"].value_counts().reindex(["<30", ">30", "NO"]).rename("n").to_frame())
dist["percent"] = (dist["n"] / dist["n"].sum() * 100).round(1)
dist.to_csv(f"{FIG_DIR}/1_dataset_label_distribution.csv")
n_patients = df["patient_nbr"].nunique()
single = (df.groupby("patient_nbr").size() == 1).mean()
print(f"encounter: {n_raw:,} → {len(df):,} | ผู้ป่วย {n_patients:,} คน | มาครั้งเดียว {single:.1%}")
print(dist.to_string())
print(f"กลับมาทุกช่วงเวลา (<30 + >30) = {dist.loc[['<30', '>30'], 'percent'].sum():.1f}%")

fig, ax = plt.subplots(figsize=(3.2, 5))
bottom = 0
for lab, color, th in [("NO", "#d1d5db", "ไม่กลับมา"), (">30", "#4b5563", "กลับมาหลัง 30 วัน"),
                       ("<30", "#1f2937", "กลับมาภายใน 30 วัน")]:
    v = dist.loc[lab, "percent"]
    ax.bar(0, v, bottom=bottom, color=color, width=0.6)
    ax.text(0, bottom + v / 2, f"{th}\n{v:.1f}%", ha="center", va="center",
            color="black" if lab == "NO" else "white", fontsize=10)
    bottom += v
ax.set_xlim(-0.5, 0.5); ax.set_xticks([]); ax.set_ylabel("% ของ encounter"); ax.set_ylim(0, 100)
ax.set_title(f"n = {len(df):,} encounter")
save(fig, "1_dataset_label_distribution")

## 2. สไลด์วิธีการ 1 — จำนวนในแต่ละ split (แบ่งตามผู้ป่วย)

In [ ]:
split = read("split_assignment.csv")
split_tab = (split.groupby("split").agg(encounters=("encounter_id", "size"), patients=("patient_nbr", "nunique"))
             .reindex(["train", "val", "test"]))
split_tab["% encounter"] = (split_tab["encounters"] / split_tab["encounters"].sum() * 100).round(1)
assert split.groupby("patient_nbr")["split"].nunique().max() == 1, "มีผู้ป่วยอยู่มากกว่า 1 split"
s = split.sort_values(["patient_nbr", "encounter_id"])
s["visit"] = s.groupby("patient_nbr").cumcount() + 1
n_test_hist = int(((s["split"] == "test") & (s["visit"] >= 2)).sum())
split_tab.to_csv(f"{FIG_DIR}/2_split_counts.csv")
print(split_tab.to_string())
print(f"✓ ไม่มีผู้ป่วยอยู่ข้าม split | test ที่มีประวัติ (visit ≥ 2) = {n_test_hist:,} encounter")

## 3. สไลด์ผลที่ 1 — Stacking: นิยาม 30 วัน vs ทุกช่วงเวลา

In [ ]:
m11 = {lab: json.load(open(f"{OUT}/11_{lab}/metrics.json")) for lab in ["30d", "any"]}
t1 = pd.DataFrame({LABEL_TH[lab]: {"ROC-AUC": m["ROC_AUC"], "PR-AUC": m["PR_AUC"],
                                     "PR-AUC เทียบการสุ่ม (เท่า)": m["PR_AUC_lift"],
                                     "Brier skill score": m["Brier_skill_score"], "base rate": m["base_rate"]}
                   for lab, m in m11.items()})
t1.to_csv(f"{FIG_DIR}/3_stacking_label_comparison.csv")
print(t1.to_string())
print(f"ผลต่าง ROC-AUC = {abs(m11['any']['ROC_AUC'] - m11['30d']['ROC_AUC']):.4f}")

## 4. สไลด์ผลที่ 2 — Odds ratio (logistic regression, label any)
เพิ่ม **เส้น 95% CI** (cluster bootstrap 200 รอบ จาก notebook 12) ซึ่งในสไลด์ไม่ได้แสดง

หมายเหตุ: สไลด์เลือก 6 ปัจจัยที่ตีความทางคลินิกได้ ไม่ใช่ 6 อันดับแรก — `admission_type = Unknown/Other` (OR 1.45) มีค่าสูงเป็นอันดับ 2 แต่ไม่ได้ใส่ เพราะเป็นรหัสที่ไม่ทราบประเภท

In [ ]:
ORS = [("number_inpatient (per visit, cap 5)", "เคย admit ก่อนหน้า (ต่อครั้ง)"),
       ("diabetesMed = Yes", "ได้ยาเบาหวาน"),
       ("number_emergency (per visit, cap 5)", "เคยมา ER (ต่อครั้ง)"),
       ("max_glu_serum = >300", "น้ำตาล > 300"),
       ("discharge_group = Home health", "ส่งต่อ home health"),
       ("A1Cresult = Norm", "A1C ปกติ")]
orr = read("12_features_any/B_odds_ratios.csv", index_col=0).loc[[k for k, _ in ORS]]
orr.insert(0, "label_th", [t for _, t in ORS])
orr.to_csv(f"{FIG_DIR}/4_odds_ratios_slide.csv")
print(orr.to_string())

t = orr.iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 3.6))
colors = np.where(t["OR"] > 1, C_30D, "#5b8c5a")
ax.barh(t["label_th"], t["OR"] - 1, left=1, color=colors, height=0.6)
ax.errorbar(t["OR"], range(len(t)), xerr=[t["OR"] - t["CI_low"], t["CI_high"] - t["OR"]],
            fmt="none", ecolor="black", elinewidth=1, capsize=3)
for i, (o, hi) in enumerate(zip(t["OR"], t["CI_high"])):
    ax.text(max(o, hi) + 0.02 if o > 1 else min(o, t["CI_low"].iloc[i]) - 0.02, i, r2(o),
            va="center", ha="left" if o > 1 else "right", fontsize=9)
ax.axvline(1, color="black", lw=1)
ax.set_xlabel("Odds ratio (95% CI จาก cluster bootstrap ตามผู้ป่วย)"); ax.set_xlim(0.7, 1.7)
save(fig, "4_odds_ratios")

## 5. สไลด์ผลที่ 3 — Hazard ratio ช่วง 30 วันแรก vs ระยะยาว (+ 95% CI ใหม่)

ค่าในสไลด์มาจาก `4_ph_check.csv` (notebook 13 หัวข้อ 4) = grouped Cox แบบจุดตัดเดียว 2 ตัว:
- **ช่วง 30 วันแรก** = HR ของการ readmit ภายใน 30 วัน
- **ระยะยาว** = HR ของการ readmit ภายใน H (= label any) — **รวมช่วง 30 วันแรกอยู่ด้วย** ถ้าอาจารย์ถาม ให้ตอบว่าเป็น "ภายในระยะติดตามทั้งหมด"

เดิมไม่มี CI จึงเพิ่ม cluster bootstrap ตามผู้ป่วย (`N_BOOT_HR` รอบ) ด้วยโค้ด `GroupedPH` เดียวกับ notebook 13

ผลที่ได้ (200 รอบ): ส่งต่อสถานพยาบาล ช่วง 30 วัน HR 1.62 (1.54–1.72) แต่ระยะยาว 0.96 (0.93–0.99) — **ต่ำกว่า 1 อย่างมีนัยสำคัญ**
จึงควรพูดว่า "ระยะยาวไม่เพิ่มความเสี่ยง (ต่ำกว่าเล็กน้อย)" แทน "ไม่มีผลระยะยาว" ส่วน AMA และลดขนาด metformin CI ช่วง 30 วันกว้างมาก (n น้อย)

In [ ]:
from scipy.optimize import minimize
from readmission_features import load_prepared, attach_split, add_id_groups, build_lr_design

class GroupedPH:
    """คัดลอกจาก 13_survival_models.ipynb ทุกบรรทัด"""
    def __init__(self, l2=1e-4):
        self.l2 = l2
    def _unpack(self, w):
        J = self.J
        a1, deltas, beta = w[0], w[1:J], w[J:]
        return a1 + np.concatenate([[0.0], np.cumsum(np.exp(deltas))]), deltas, beta
    def _negll(self, w, Z, c):
        J = self.J
        alphas, deltas, beta = self._unpack(w)
        eta = Z @ beta
        Lam = np.exp(np.clip(alphas[None, :] + eta[:, None], -30, 30))
        ll = np.zeros(len(c)); dLam = np.zeros_like(Lam)
        m = c == 0
        ll[m] = np.log(-np.expm1(-Lam[m, 0])); dLam[m, 0] = 1.0 / np.expm1(Lam[m, 0])
        for k in range(1, J):
            m = c == k
            Dk = Lam[m, k] - Lam[m, k - 1]
            ll[m] = -Lam[m, k - 1] + np.log(-np.expm1(-Dk))
            dLam[m, k - 1] = -1.0 - 1.0 / np.expm1(Dk); dLam[m, k] = 1.0 / np.expm1(Dk)
        m = c == J
        ll[m] = -Lam[m, J - 1]; dLam[m, J - 1] = -1.0
        dA = dLam * Lam
        g_alpha = dA.sum(axis=0); g_beta = Z.T @ dA.sum(axis=1)
        g_a1 = g_alpha.sum()
        g_delta = np.array([g_alpha[k:].sum() * np.exp(deltas[k - 1]) for k in range(1, J)])
        n = len(c)
        f = -ll.sum() / n + self.l2 * beta @ beta
        g = -np.concatenate([[g_a1], g_delta, g_beta]) / n
        g[J:] += 2 * self.l2 * beta
        return f, g
    def fit(self, Z, c, n_cuts):
        self.J, self.p = n_cuts, Z.shape[1]
        self.mu_, self.sd_ = Z.mean(0), Z.std(0) + 1e-12
        Zs = (Z - self.mu_) / self.sd_
        w0 = np.zeros(self.J + self.p)
        w0[0] = np.log(-np.log(1 - max((c == 0).mean(), 1e-3)))
        res = minimize(self._negll, w0, args=(Zs, c), jac=True, method="L-BFGS-B", options={"maxiter": 2000})
        self.converged_ = res.success
        _, _, beta_s = self._unpack(res.x)
        self.beta_ = beta_s / self.sd_
        return self

X_, _, groups, meta, readable = load_prepared(RAW_PATH, "any")
sp = attach_split(meta, f"{OUT}/split_assignment.csv")
tr = np.where(sp == "train")[0]
R = add_id_groups(readable)
D_tr = build_lr_design(R.iloc[tr])
Z = D_tr.values
cat = meta["readmitted_raw"].map({"<30": 0, ">30": 1, "NO": 2}).values[tr]
c30, cH = np.where(cat == 0, 0, 1), np.where(cat <= 1, 0, 1)
g_tr = groups[tr]

HRS = [("discharge_group = Facility", "ส่งต่อสถานพยาบาล"), ("discharge_group = Left AMA", "ออก รพ. เอง (AMA)"),
       ("metformin = Down", "ลดขนาด metformin"), ("number_inpatient (per visit, cap 5)", "inpatient ก่อนหน้า"),
       ("number_emergency (per visit, cap 5)", "ER ก่อนหน้า"), ("diabetesMed = Yes", "ได้ยาเบาหวาน")]
cols = list(D_tr.columns)
ix = [cols.index(k) for k, _ in HRS]

b30 = GroupedPH().fit(Z, c30, 1).beta_[ix]
bH = GroupedPH().fit(Z, cH, 1).beta_[ix]
ref = read("13_survival/4_ph_check.csv", index_col=0).loc[[k for k, _ in HRS]]
assert np.allclose(np.exp(b30).round(3), ref["HR_30d"], atol=2e-3), "point estimate ไม่ตรงกับ notebook 13"
assert np.allclose(np.exp(bH).round(3), ref["HR_H"], atol=2e-3), "point estimate ไม่ตรงกับ notebook 13"
print("✓ ค่า HR ตรงกับ 4_ph_check.csv — เริ่ม bootstrap")

rng = np.random.default_rng(RANDOM_STATE)
uniq = np.unique(g_tr)
rows_by_pt = pd.Series(np.arange(len(g_tr))).groupby(g_tr).apply(np.array)
boot30, bootH = [], []
for b in range(N_BOOT_HR):
    ii = np.concatenate(rows_by_pt.loc[rng.choice(uniq, len(uniq), replace=True)].values)
    boot30.append(GroupedPH().fit(Z[ii], c30[ii], 1).beta_[ix])
    bootH.append(GroupedPH().fit(Z[ii], cH[ii], 1).beta_[ix])
    if (b + 1) % 25 == 0:
        print(f"  bootstrap {b + 1}/{N_BOOT_HR}")
boot30, bootH = np.array(boot30), np.array(bootH)

hr = pd.DataFrame({"label_th": [t for _, t in HRS],
                   "HR_30d": np.exp(b30), "HR_30d_low": np.exp(np.percentile(boot30, 2.5, 0)),
                   "HR_30d_high": np.exp(np.percentile(boot30, 97.5, 0)),
                   "HR_H": np.exp(bH), "HR_H_low": np.exp(np.percentile(bootH, 2.5, 0)),
                   "HR_H_high": np.exp(np.percentile(bootH, 97.5, 0))}, index=[k for k, _ in HRS])
for p in ["30d", "H"]:
    hr[f"sig_{p}"] = (hr[f"HR_{p}_low"] > 1) | (hr[f"HR_{p}_high"] < 1)
hr = hr.round(3)
hr.to_csv(f"{FIG_DIR}/5_hazard_ratios_30d_vs_H_ci.csv")
print(hr.to_string())

t = hr.iloc[::-1]
y = np.arange(len(t))
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for off, p, color, lab in [(0.18, "30d", C_30D, "ช่วง 30 วันแรก"), (-0.18, "H", C_ANY, "ระยะยาว (ภายใน H)")]:
    v = t[f"HR_{p}"]
    ax.barh(y + off, v - 1, left=1, height=0.34, color=color, label=lab)
    ax.errorbar(v, y + off, xerr=[(v - t[f"HR_{p}_low"]).clip(lower=0), (t[f"HR_{p}_high"] - v).clip(lower=0)], fmt="none",
                ecolor="black", elinewidth=0.8, capsize=2)
    for yy, vv, hi in zip(y + off, v, t[f"HR_{p}_high"]):
        ax.text(hi + 0.02, yy, r2(vv), va="center", fontsize=8)
ax.axvline(1, color="black", lw=1, ls="--")
ax.set_yticks(y); ax.set_yticklabels(t["label_th"])
ax.set_xlabel("Hazard ratio (>1 = เสี่ยงเพิ่ม), 95% CI จาก cluster bootstrap"); ax.legend(loc="lower right")
save(fig, "5_hazard_ratios_30d_vs_H")

## 6. สไลด์ผลที่ 4 — Ablation A–D (test)

In [ ]:
abl = {lab: read(f"14_lstm_{lab}/test_metrics.csv", index_col=0) for lab in ["any", "30d"]}
names = ["A: XGBoost\n(ปรับจูน)", "B: XGBoost\n+ ประวัติ", "C: LSTM\n(K = 1)", "D: LSTM\n(K = 5)"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, col, title in [(axes[0], "ROC_AUC", f"ผู้ป่วยทั้งหมด (n = {split_tab.loc['test', 'encounters']:,})"),
                       (axes[1], "ROC_AUC_has_history", f"ผู้ป่วยที่มีประวัติ (n = {n_test_hist:,})")]:
    x = np.arange(4)
    for off, lab, color in [(-0.2, "any", C_ANY), (0.2, "30d", C_30D)]:
        v = abl[lab][col].values
        ax.bar(x + off, v, width=0.4, color=color, label=LABEL_TH[lab])
        for xx, vv in zip(x + off, v):
            ax.text(xx, vv + 0.001, f"{vv:.4f}", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(names); ax.set_title(title, pad=28); ax.set_ylim(0.62, 0.71)
axes[0].set_ylabel("ROC-AUC (ชุดทดสอบ)")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.0))
save(fig, "6_ablation")
pd.concat(abl, names=["label"]).to_csv(f"{FIG_DIR}/6_ablation.csv")
print(pd.concat({lab: read(f"14_lstm_{lab}/test_deltas.csv") for lab in ["any", "30d"]}).to_string())

## 7. สไลด์แบบจำลองสุดท้าย — ROC-AUC 6 โมเดล + 95% CI

Survival = XGBoost AFT (H = 365) จาก `13_survival/summary.json` (ค่า 30 วัน = P(T ≤ 30) ของโมเดลตัวเดียวกัน)

**CI ของโมเดลสุดท้ายเดิมไม่มีในไฟล์** (อยู่แค่ใน output ของ notebook 15) จึงคำนวณใหม่:
score ของโมเดลสุดท้าย = w·D + (1−w)·B จาก `14_lstm_*/test_predictions.csv` ใช้ bootstrap ตามผู้ป่วย 500 รอบ seed 42 แบบเดียวกับ notebook 15

In [ ]:
surv = json.load(open(f"{OUT}/13_survival/summary.json"))["point_metrics"]["XGBoost AFT (H=365)"]
final = {}
for lab in ["any", "30d"]:
    m15 = read(f"15_final_{lab}/test_metrics.csv", index_col=0)
    fin_name = [i for i in m15.index if i.startswith("FINAL")][0]
    w = float(re.search(r"w_lstm=([0-9.]+)", fin_name).group(1))   # อ่านจากชื่อ (blend_selection.csv ปัดทศนิยมจนค่าเสมอกันได้)
    p14 = read(f"14_lstm_{lab}/test_predictions.csv")
    score = w * p14["D. LSTM K=5"].values + (1 - w) * p14["B. XGBoost tuned + history"].values
    yt = p14["y_true"].values
    g = p14[["encounter_id"]].merge(split[["encounter_id", "patient_nbr"]], on="encounter_id")["patient_nbr"].values
    auc = roc_auc_score(yt, score)
    assert abs(auc - m15.loc[fin_name, "ROC_AUC"]) < 5e-4, f"สร้าง score โมเดลสุดท้ายใหม่ไม่ตรง ({lab})"
    rng = np.random.default_rng(RANDOM_STATE)
    uq = np.unique(g)
    by_pt = pd.Series(np.arange(len(g))).groupby(g).apply(np.array)
    boots = [roc_auc_score(yt[ii], score[ii])
             for ii in (np.concatenate(by_pt.loc[rng.choice(uq, len(uq), replace=True)].values)
                        for _ in range(N_BOOT_TEST))]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    final[lab] = {"w_lstm": w, "ROC_AUC": auc, "CI_low": lo, "CI_high": hi,
                  "Stacking": m15.loc["Stacking (notebook 11)", "ROC_AUC"],
                  "Survival (XGBoost AFT)": surv[f"ROC_AUC_{lab}"],
                  "A. XGBoost tuned": m15.loc["A. XGBoost tuned", "ROC_AUC"],
                  "B. XGBoost + history": m15.loc["B. XGBoost + history", "ROC_AUC"],
                  "D. LSTM K=5": m15.loc["D. LSTM K=5", "ROC_AUC"]}
final_tab = pd.DataFrame(final).T.round(4)
final_tab.to_csv(f"{FIG_DIR}/7_final_model_comparison_ci.csv")
print(final_tab.to_string())

order = ["Stacking", "Survival (XGBoost AFT)", "A. XGBoost tuned", "B. XGBoost + history", "D. LSTM K=5", "ROC_AUC"]
names = ["Stacking\n(baseline)", "Survival\n(XGBoost AFT)", "A: XGBoost\n(ปรับจูน)", "B: XGBoost\n+ ประวัติ",
         "D: LSTM\n(K = 5)", "แบบจำลองสุดท้าย\n(ผสม B + D)"]
fig, ax = plt.subplots(figsize=(11, 4.8))
x = np.arange(len(order))
for off, lab, color in [(-0.2, "any", C_ANY), (0.2, "30d", C_30D)]:
    v = final_tab.loc[lab, order].values.astype(float)
    ax.bar(x + off, v, width=0.4, color=color, label=LABEL_TH[lab])
    r = final_tab.loc[lab]
    for i, (xx, vv) in enumerate(zip(x + off, v)):
        top = r["CI_high"] if i == len(order) - 1 else vv    # ตัวสุดท้ายวางเลขเหนือเส้น CI
        ax.text(xx, top + 0.001, f"{vv:.4f}", ha="center", fontsize=8)
    ax.errorbar(x[-1] + off, r["ROC_AUC"], yerr=[[r["ROC_AUC"] - r["CI_low"]], [r["CI_high"] - r["ROC_AUC"]]],
                fmt="none", ecolor="black", capsize=3)
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylim(0.64, 0.72)
ax.set_ylabel("ROC-AUC (ชุดทดสอบ)"); ax.legend(loc="upper left", ncol=2, frameon=False)
ax.set_title("เส้นดำ = 95% CI ของแบบจำลองสุดท้าย (bootstrap ตามผู้ป่วย 500 รอบ)", fontsize=9)
save(fig, "7_final_model_comparison")
for lab in ["any", "30d"]:
    print(f"{LABEL_TH[lab]}: Δ vs stacking = {final_tab.loc[lab, 'ROC_AUC'] - final_tab.loc[lab, 'Stacking']:+.4f}")

## 8. สไลด์ระดับความเสี่ยง — อัตรากลับมาจริงในแต่ละระดับ (test)

In [ ]:
tiers = {lab: read(f"15_final_{lab}/risk_tiers.csv", index_col=0) for lab in ["any", "30d"]}
base = {lab: m11[lab]["base_rate"] for lab in ["any", "30d"]}
tier_tab = pd.DataFrame({
    "% ผู้ป่วย": tiers["30d"]["% of patients"].round(1),
    "ทุกช่วงเวลา: อัตราจริง %": (tiers["any"]["observed_rate"] * 100).round(1),
    "ทุกช่วงเวลา: ทำนายเฉลี่ย %": (tiers["any"]["mean_predicted"] * 100).round(1),
    "30 วัน: อัตราจริง %": (tiers["30d"]["observed_rate"] * 100).round(1),
    "30 วัน: ทำนายเฉลี่ย %": (tiers["30d"]["mean_predicted"] * 100).round(1),
    "30 วัน: % ของผู้กลับมาทั้งหมด": tiers["30d"]["% of all readmissions captured"].round(1),
})
tier_tab.loc["ค่าเฉลี่ย"] = [100, round(base["any"] * 100, 1), np.nan, round(base["30d"] * 100, 1), np.nan, 100]
tier_tab.to_csv(f"{FIG_DIR}/8_risk_tiers.csv")
print(tier_tab.to_string())
gap = max((tiers[l]["observed_rate"] - tiers[l]["mean_predicted"]).abs().max() for l in tiers)
print(f"\nค่าทำนายต่างจากอัตราจริงสูงสุด = {gap:.4f}")
print(f"30 วัน: กลุ่มสูงจับผู้ที่กลับมาได้ {tiers['30d'].loc['High', '% of all readmissions captured']:.1f}%, "
      f"lift = {tiers['30d'].loc['High', 'lift vs base']:.2f} เท่า")

## 9. สไลด์ความเป็นธรรม — ผู้สูงอายุ 90+

In [ ]:
fair = {lab: read(f"15_final_{lab}/fairness_age.csv").set_index("group") for lab in ["any", "30d"]}
overall = {lab: final_tab.loc[lab, "ROC_AUC"] for lab in ["any", "30d"]}
fair_tab = pd.DataFrame({lab: {"ROC-AUC ทั้งหมด": overall[lab],
                               "ROC-AUC อายุ 90–100": fair[lab].loc["[90-100)", "ROC_AUC"],
                               "n อายุ 90–100": fair[lab].loc["[90-100)", "n"],
                               "% ถูกจัดเป็น High (90–100)": fair[lab].loc["[90-100)", "% High"],
                               "recall ของ High (90–100)": fair[lab].loc["[90-100)", "recall(High)"],
                               "ROC-AUC ต่ำสุด–สูงสุด อายุ 40–80":
                                   f"{fair[lab].loc['[40-50)':'[70-80)', 'ROC_AUC'].min():.3f}–"
                                   f"{fair[lab].loc['[40-50)':'[70-80)', 'ROC_AUC'].max():.3f}"}
                         for lab in ["any", "30d"]})
fair_tab.to_csv(f"{FIG_DIR}/9_fairness_age90.csv")
print(fair_tab.to_string())

fig, ax = plt.subplots(figsize=(6, 4.5))
x = np.arange(2)
v_all = [overall["any"], overall["30d"]]
v_90 = [fair["any"].loc["[90-100)", "ROC_AUC"], fair["30d"].loc["[90-100)", "ROC_AUC"]]
ax.bar(x - 0.2, v_all, 0.4, color=C_ANY, label="ผู้ป่วยทั้งหมด")
ax.bar(x + 0.2, v_90, 0.4, color=C_30D, label="อายุ 90 ปีขึ้นไป")
for xx, vv in list(zip(x - 0.2, v_all)) + list(zip(x + 0.2, v_90)):
    ax.text(xx, vv + 0.005, f"{vv:.3f}", ha="center", fontsize=9, fontweight="bold")
ax.axhline(0.5, ls="--", color="grey", lw=1); ax.text(-0.55, 0.505, "เดาสุ่ม", color="grey", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(["กลับมาเมื่อไรก็ได้ (any)", "ภายใน 30 วัน"])
ax.set_ylim(0.45, 0.75); ax.set_ylabel("ROC-AUC"); ax.legend()
save(fig, "9_fairness_age90")

## 10. ตรวจตัวเลขในสไลด์เทียบกับไฟล์
ค่าใน `SLIDE` คัดลอกจากสไลด์ ถ้าแก้สไลด์ให้แก้ตรงนี้ด้วย แล้วรันใหม่เพื่อตรวจ

In [ ]:
SLIDE = [  # (สไลด์, รายการ, ค่าที่เขียนในสไลด์, ค่าจากไฟล์, ทศนิยมที่ปัดในสไลด์)
    ("Dataset", "encounter หลังทำความสะอาด", 99340, len(df), 0),
    ("Dataset", "% กลับมาภายใน 30 วัน", 11.4, dist.loc["<30", "percent"], 1),
    ("Dataset", "% กลับมาหลัง 30 วัน", 35.7, dist.loc[">30", "percent"], 1),
    ("Dataset", "% ไม่กลับมา", 52.9, dist.loc["NO", "percent"], 1),
    ("วิธีการ 1", "train", 63605, split_tab.loc["train", "encounters"], 0),
    ("วิธีการ 1", "val", 15962, split_tab.loc["val", "encounters"], 0),
    ("วิธีการ 1", "test", 19773, split_tab.loc["test", "encounters"], 0),
    ("ผลที่ 1", "ROC-AUC 30 วัน", 0.6745, m11["30d"]["ROC_AUC"], 4),
    ("ผลที่ 1", "ROC-AUC ทุกช่วงเวลา", 0.6806, m11["any"]["ROC_AUC"], 4),
    ("ผลที่ 1", "PR-AUC 30 วัน", 0.2146, m11["30d"]["PR_AUC"], 4),
    ("ผลที่ 1", "PR-AUC ทุกช่วงเวลา", 0.6365, m11["any"]["PR_AUC"], 4),
    ("ผลที่ 1", "Brier skill 30 วัน", 0.047, m11["30d"]["Brier_skill_score"], 3),
    ("ผลที่ 1", "Brier skill ทุกช่วงเวลา", 0.100, m11["any"]["Brier_skill_score"], 3),
    *[("ผลที่ 2", f"OR {th}", v, orr.loc[k, "OR"], 2) for (k, th), v in zip(ORS, [1.51, 1.37, 1.29, 1.26, 1.16, 0.83])],
    *[("ผลที่ 3", f"HR 30 วัน {th}", v, hr.loc[k, "HR_30d"], 2)
      for (k, th), v in zip(HRS, [1.62, 1.42, 1.36, 1.33, 1.09, 1.16])],
    *[("ผลที่ 3", f"HR ระยะยาว {th}", v, hr.loc[k, "HR_H"], 2)
      for (k, th), v in zip(HRS, [0.96, 1.05, 1.06, 1.28, 1.15, 1.25])],
    ("ผลที่ 3", "AFT ROC-AUC any", 0.6937, surv["ROC_AUC_any"], 4),
    ("ผลที่ 3", "XGBoost tuned ROC-AUC any", 0.6946, abl["any"].loc["A. XGBoost tuned", "ROC_AUC"], 4),
    *[("ผลที่ 4", f"{lab} {c} {mdl[:2]}", v, abl[lab].loc[mdl, c], 4)
      for lab, c, vals in [("any", "ROC_AUC", [0.6946, 0.6969, 0.6981, 0.7014]),
                           ("30d", "ROC_AUC", [0.6795, 0.6845, 0.6899, 0.6911]),
                           ("any", "ROC_AUC_has_history", [0.6663, 0.6792, 0.6706, 0.6857]),
                           ("30d", "ROC_AUC_has_history", [0.6318, 0.6546, 0.6596, 0.6642])]
      for mdl, v in zip(abl[lab].index, vals)],
    ("ผลที่ 4", "n ผู้ป่วยที่มีประวัติ", 5775, n_test_hist, 0),
    ("สุดท้าย", "ROC-AUC any", 0.7031, final_tab.loc["any", "ROC_AUC"], 4),
    ("สุดท้าย", "CI any ต่ำ", 0.694, final_tab.loc["any", "CI_low"], 3),
    ("สุดท้าย", "CI any สูง", 0.711, final_tab.loc["any", "CI_high"], 3),
    ("สุดท้าย", "ROC-AUC 30 วัน", 0.6918, final_tab.loc["30d", "ROC_AUC"], 4),
    ("สุดท้าย", "CI 30 วัน ต่ำ", 0.679, final_tab.loc["30d", "CI_low"], 3),
    ("สุดท้าย", "CI 30 วัน สูง", 0.704, final_tab.loc["30d", "CI_high"], 3),
    ("สุดท้าย", "Survival 30 วัน", 0.6548, surv["ROC_AUC_30d"], 4),
    ("สุดท้าย", "Δ vs stacking any", 0.023, final_tab.loc["any", "ROC_AUC"] - final_tab.loc["any", "Stacking"], 3),
    ("สุดท้าย", "Δ vs stacking 30 วัน", 0.017, final_tab.loc["30d", "ROC_AUC"] - final_tab.loc["30d", "Stacking"], 3),
    *[("ระดับความเสี่ยง", f"{lab} {tier}", v, tiers[lab].loc[tier, "observed_rate"] * 100, 1)
      for lab, vals in [("any", [73.4, 53.3, 32.2]), ("30d", [24.1, 11.8, 6.2])]
      for tier, v in zip(["High", "Medium", "Low"], vals)],
    ("ระดับความเสี่ยง", "30 วัน กลุ่มสูงจับได้ %", 41, tiers["30d"].loc["High", "% of all readmissions captured"], 0),
    ("ระดับความเสี่ยง", "30 วัน lift", 2.1, tiers["30d"].loc["High", "lift vs base"], 1),
    ("ระดับความเสี่ยง", "ค่าทำนายต่างจากจริงไม่เกิน", 0.02, gap, 2),
    ("ความเป็นธรรม", "n 90–100", 547, fair["any"].loc["[90-100)", "n"], 0),
    ("ความเป็นธรรม", "ROC-AUC 90+ any", 0.646, fair["any"].loc["[90-100)", "ROC_AUC"], 3),
    ("ความเป็นธรรม", "ROC-AUC 90+ 30 วัน", 0.553, fair["30d"].loc["[90-100)", "ROC_AUC"], 3),
    ("ความเป็นธรรม", "% High 90+ any", 9.5, fair["any"].loc["[90-100)", "% High"], 1),
    ("ความเป็นธรรม", "recall 90+ any %", 15, fair["any"].loc["[90-100)", "recall(High)"] * 100, 0),
    ("ความเป็นธรรม", "% High 90+ 30 วัน", 16.6, fair["30d"].loc["[90-100)", "% High"], 1),
    ("ความเป็นธรรม", "recall 90+ 30 วัน %", 23.5, fair["30d"].loc["[90-100)", "recall(High)"] * 100, 1),
]
chk = pd.DataFrame(SLIDE, columns=["สไลด์", "รายการ", "ในสไลด์", "จากไฟล์", "ทศนิยม"])
chk["จากไฟล์"] = chk["จากไฟล์"].astype(float)
chk["ตรง"] = [abs(f - s) <= 0.5 * 10 ** -d + 1e-9 if r != "ค่าทำนายต่างจากจริงไม่เกิน" else f <= s + 1e-9
              for r, s, f, d in zip(chk["รายการ"], chk["ในสไลด์"], chk["จากไฟล์"], chk["ทศนิยม"])]
chk.to_csv(f"{FIG_DIR}/10_slide_number_check.csv", index=False)
print(f"ตรงกัน {chk['ตรง'].sum()} / {len(chk)} รายการ")
if not chk["ตรง"].all():
    print("\nรายการที่ไม่ตรง:")
    print(chk[~chk["ตรง"]].to_string(index=False))
print(f"\nบันทึกรูปและตารางไว้ที่ {FIG_DIR}/")